# Pandas — мини-проект

Сквозной мини-проект на реальных данных. Повторяет ключевые темы урока.


In [ ]:
# Практика к уроку — выполняйте ячейки по порядку
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)



## Постановка задачи `(intro)`


Разберём **Titanic (OpenML)**: загрузка, очистка, groupby, merge, pivot, экспорт инсайтов и подготовка данных для sklearn Pipeline.


## Загрузка и первичный осмотр `(eda)`


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml

np.random.seed(42)
sns.set_theme(style="whitegrid", font_scale=1.05)

raw = fetch_openml("titanic", version=1, as_frame=True, parser="auto")
df = raw.frame.copy()
df["Survived"] = df["survived"].astype(int)
df["Pclass"] = df["pclass"].astype(int)
df["Age"] = pd.to_numeric(df["age"], errors="coerce")
df["Fare"] = pd.to_numeric(df["fare"], errors="coerce")
df["Sex"] = df["sex"].astype(str)
df["Embarked"] = df["embarked"].astype(str)

print(f"Объектов: {len(df)}")
display(df.head())
print("Пропуски:\n", df[["Age", "Fare", "Embarked"]].isna().sum())



## Очистка и типы `(example)`


In [ ]:
df_clean = df.copy()
df_clean["Age"] = df_clean["Age"].fillna(df_clean.groupby("Pclass")["Age"].transform("median"))
df_clean["Embarked"] = df_clean["Embarked"].fillna(df_clean["Embarked"].mode()[0])
df_clean["Sex"] = df_clean["Sex"].astype("category")
print("Пропуски Age после impute:", df_clean["Age"].isna().sum())



## groupby и агрегаты `(viz)`


In [ ]:
summary = df_clean.groupby(["Pclass", "Sex"]).agg(
    count=("Survived", "size"),
    survival_rate=("Survived", "mean"),
    avg_fare=("Fare", "mean"),
).round(3)
display(summary)

summary.reset_index().pivot(index="Pclass", columns="Sex", values="survival_rate").plot(
    kind="bar", figsize=(7, 4), title="Доля выживших по классу и полу"
)
plt.ylabel("survival_rate")
plt.tight_layout()
plt.show()



## merge справочника `(example)`


In [ ]:
ports = pd.DataFrame({
    "Embarked": ["S", "C", "Q"],
    "port_name": ["Southampton", "Cherbourg", "Queenstown"],
})
df_ports = pd.merge(df_clean, ports, on="Embarked", how="left")
df_ports.groupby("port_name")["Survived"].mean().sort_values(ascending=False)



## pivot_table и инсайты `(viz)`


In [ ]:
pt = df_clean.pivot_table(values="Survived", index="Pclass", columns="Sex", aggfunc="mean")
display(pt.round(2))
print("Инсайт: женщины 1 класса выживали чаще всего")



## Экспорт и Pipeline `(example)`


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

feature_cols = ["Pclass", "Age", "Fare", "Sex", "Embarked"]
X = df_clean[feature_cols]
y = df_clean["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), ["Pclass", "Age", "Fare"]),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), ["Sex", "Embarked"]),
])
pipe = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=500, random_state=42))])
pipe.fit(X_train, y_train)
print(f"Test accuracy: {pipe.score(X_test, y_test):.3f}")

insights = summary.reset_index()
insights.to_csv("titanic_groupby_summary.csv", index=False)
print("Сводка сохранена в titanic_groupby_summary.csv")



## Чек-лист мини-проекта `(summary)`


1. `fetch_openml` → `head/info/describe`.
2. Пропуски — осмысленный impute (по группе или median).
3. groupby + pivot — инсайты до модели.
4. merge справочников по ключу.
5. Pipeline после train_test_split.
6. Экспорт агрегатов для отчёта.
